# AGFN denovo: molecule + docking visualization

Three views of what the model is generating:

1. **Trajectory step grids** — partial molecules at each GFN action step (RDKit 2D).
2. **Top-K across training** — best generated molecules ranked by reward or Uni-Dock affinity, from `<log_dir>/top_k_mols.pt` produced by the `TopKTracker` hook in `samp_iter_finetune.py`.
3. **3D docked pose** — a fresh batch redocked into a persistent directory, rendered with py3Dmol.

**Run from the repo root.** All paths in `denovo.yml` are relative; cell 2 cd's to the repo root for you. Requires `py3Dmol` (in `requirements_jn.txt`).

In [ ]:
# ----- User parameters (edit these) -----
CONFIG_PATH      = "./src/config/denovo.yml"
SAVED_MODEL_PATH = None     # None -> use hps.saved_model_path from YAML
TARGET_NAME      = None     # None -> use hps.target_name from YAML
N_SAMPLES        = 8        # trajectories to draw
N_DOCK           = 4        # subset to dock
TOP_K_SHOW       = 20       # rows of top-K to render (max 100)
TOP_K_RANK_BY    = "reward" # "reward" or "affinity"
PERSIST_DIR      = "./AGFN_logs/visualize_out"
DEVICE_ID        = 0
SEED             = 0

In [ ]:
import os, sys, json, yaml
from pathlib import Path

# chdir to repo root so relative paths in denovo.yml resolve as they do for denovo_driver.py
_here = Path.cwd()
for cand in [_here, *_here.parents]:
    if (cand / "src" / "config" / "denovo.yml").exists():
        os.chdir(cand); break
REPO_ROOT = Path.cwd()
print("repo root:", REPO_ROOT)

for p in ("src", "src/apps/docking", "src/apps/docking/denovo"):
    full = str((REPO_ROOT / p).resolve())
    if full not in sys.path: sys.path.insert(0, full)

import numpy as np
import torch
from easydict import EasyDict
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, AllChem
from rdkit.Chem.Draw import IPythonConsole
import py3Dmol
RDLogger.DisableLog("rdApp.*")
torch.manual_seed(SEED); np.random.seed(SEED)

In [ ]:
# Load + patch hps for the QedxSaxDock denovo task.
with open(CONFIG_PATH) as f:
    hps = EasyDict(yaml.safe_load(f)).finetuning
hps.update({"Z_learning_rate": 1e-3})

if hps.task == "QedxSaxDock":
    conditional_range_dict = {
        "tpsa":      [[10, 200], [10, 200], 0],
        "num_rings": [[1, 5],   [1, 5],   1],
        "sas":       [[1, 5],   [1, 5],   0],
        "qed":       [[0.5, 1], [0, 1],   0],
    }
else:
    raise RuntimeError(f"unsupported task: {hps.task}")
cond_prop_var = {"tpsa": 20, "num_rings": 1, "sas": 1, "qed": 1}
hps.task_conditionals = False
hps.task_rewards_only = False
hps.update(conditional_range_dict)

if SAVED_MODEL_PATH is not None: hps.saved_model_path = SAVED_MODEL_PATH
if TARGET_NAME      is not None: hps.target_name      = TARGET_NAME

gfn_samples_path = f"{hps.gfn_samples_path}/GFN_gen_samples_{hps.target_name}/"
os.makedirs(gfn_samples_path, exist_ok=True)
os.makedirs(PERSIST_DIR, exist_ok=True)

print("target_name:     ", hps.target_name)
print("saved_model_path:", hps.saved_model_path)
print("log_dir:         ", hps.log_dir)
print("gfn_samples_path:", gfn_samples_path)

In [ ]:
# Build DockingFineTuner and load checkpoint. Reuses the exact constructor
# used by training, so model architecture and checkpoint loading match.
from denovo_trainer import DockingFineTuner

finetuner = DockingFineTuner(
    hps, conditional_range_dict, cond_prop_var,
    hps.saved_model_path,
    rank=DEVICE_ID, world_size=1,
    gfn_samples_path=gfn_samples_path,
)
finetuner.gfn_trainer.model.train(False)  # PyTorch evaluation mode
finetuner.gfn_trainer.model_prior.train(False)
print("device:", finetuner.device)

In [ ]:
# Sample N_SAMPLES trajectories. Mirrors samp_iter_finetune.py:217-224.
ci = finetuner.cond_info_task
cond_info     = ci.compute_cond_info_forward(N_SAMPLES)
cond_info_enc = ci.thermometer_encoding(cond_info).to(finetuner.device)
with torch.no_grad():
    trajs = finetuner.graph_sampler.sample_from_model(
        finetuner.gfn_trainer.model, N_SAMPLES,
        cond_info_enc, finetuner.device,
        random_stop_action_prob=0.0,
        random_action_prob=0.0,
        seed_graph=None,
    )
valid = [t for t in trajs if t["is_valid"]]
print(f"sampled {len(trajs)} trajectories, {len(valid)} valid")

In [ ]:
# Trajectory step grids. traj[t][0] is already the pre-action Graph state —
# no need to call env.step to reconstruct intermediates.
from IPython.display import display

for idx, t in enumerate(valid[:3]):
    steps = []
    for g, _action in t["traj"]:
        if len(g.nodes) == 0:
            continue
        try:
            m = finetuner.ctx.graph_to_mol(g)
        except Exception:
            m = None
        if m is not None:
            steps.append(m)
    if not steps:
        print(f"trajectory {idx}: no renderable intermediate graphs")
        continue
    print(f"trajectory {idx}  steps={len(steps)}")
    legends = [f"t={i}" for i in range(len(steps))]
    display(Draw.MolsToGridImage(steps, molsPerRow=6, subImgSize=(220, 180), legends=legends))

In [ ]:
# Top-K best generated molecules across the whole training run.
# Produced by TopKTracker hook in src/iterators/samp_iter_finetune.py.
top_path = Path(hps.log_dir) / "top_k_mols.pt"
if not top_path.exists():
    print(f"WARNING: no top-K dump at {top_path}.")
    print("Re-run training (denovo_driver.py) so the TopKTracker hook can write it.")
    print("Saves trigger every hps.checkpoint_every iters; defaults to 10.")
else:
    top = torch.load(top_path, map_location="cpu")
    bucket_key = f"top_by_{TOP_K_RANK_BY}"
    available_ks = sorted(top.get(bucket_key, {}).keys())
    if not available_ks:
        print(f"top_k_mols.pt has no bucket '{bucket_key}'")
    else:
        k = max(k for k in available_ks if k <= max(available_ks))
        bucket = top[bucket_key][k]
        n_show = min(TOP_K_SHOW, len(bucket))
        rows = bucket[:n_show]
        mols, legends = [], []
        for smiles, reward, affinity, iteration in rows:
            m = Chem.MolFromSmiles(smiles)
            if m is None: continue
            aff_str = "  n/a" if affinity is None else f"{affinity:+.2f}"
            mols.append(m)
            legends.append(f"r={reward:+.2f}  aff={aff_str}  it={iteration}")
        print(f"showing top {len(mols)} of {len(bucket)} ranked by {TOP_K_RANK_BY}")
        display(Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(260, 200), legends=legends))

In [ ]:
# Extract final SMILES from the live sample for re-docking.
smiles, final_mols = [], []
for t in valid[:N_DOCK]:
    try:
        m = finetuner.ctx.graph_to_mol(t["traj"][-1][0])
        s = Chem.MolToSmiles(m) if m is not None else None
    except Exception:
        s, m = None, None
    if s:
        smiles.append(s); final_mols.append(m)
for i, s in enumerate(smiles):
    print(f"  [{i}] {s}")
print(f"will dock {len(smiles)} molecules")

In [ ]:
# Dock the batch with Uni-Dock (in-process). Pose .sdf files persist under PERSIST_DIR
# so the 3D viewer below can read them. Mirrors the training backend (src/apps/docking/unidock.py).
from pathlib import Path
from unidock import run_etkdg_func  # same ETKDG embedder the reward backend uses
from unidock_tools.application.unidock_pipeline import UniDock

REWARD_SCALE_MAX, REWARD_SCALE_MIN = -1.0, -10.0
grid = dict(hps.target_grid[hps.target_name])
receptor = grid["receptor"]
center = (grid["center_x"], grid["center_y"], grid["center_z"])
size   = (grid["size_x"],   grid["size_y"],   grid["size_z"])

dock_dir  = Path(PERSIST_DIR) / f"{hps.target_name}_unidock"
etkdg_dir = dock_dir / "etkdg"
save_dir  = dock_dir / "poses"
etkdg_dir.mkdir(parents=True, exist_ok=True)
save_dir.mkdir(parents=True, exist_ok=True)

sdf_inputs = [f for f in (run_etkdg_func((s, etkdg_dir / f"{i}.sdf"))
                          for i, s in enumerate(smiles)) if f is not None]
if sdf_inputs:
    UniDock(Path(receptor), sdf_inputs, *center, *size, dock_dir / "workdir").docking(
        save_dir, num_modes=1,
        search_mode=hps.get("unidock_search_mode", "fast"), seed=SEED,
    )

# Parse docking_score per ligand; clamp to <= 0 and scale exactly as the reward backend.
smiles_out, affinities, rewards, pose_paths = [], [], [], []
for i, s in enumerate(smiles):
    pose = save_dir / f"{i}.sdf"
    try:
        m = list(Chem.SDMolSupplier(str(pose)))[0]
        aff = min(float(m.GetProp("docking_score")), 0.0)
        ok = m is not None
    except Exception:
        aff, ok = 0.0, False
    rew = (aff + REWARD_SCALE_MIN) / (REWARD_SCALE_MIN + REWARD_SCALE_MAX) - 1
    smiles_out.append(s); affinities.append(aff); rewards.append(rew)
    pose_paths.append(str(pose) if ok else None)

print("results")
print(f"  poses: {save_dir}")
for i, (s, a, r) in enumerate(zip(smiles_out, affinities, rewards)):
    print(f"  [{i}] aff={a:+.2f}  reward={float(r):+.3f}  smi={s}")

In [ ]:
# 3D pose viewer. Receptor rainbow cartoon + ligand green-carbon sticks.
def view_pose(receptor_pdbqt, ligand_sdf, w=500, h=400):
    with open(receptor_pdbqt) as f: rec = f.read()
    with open(ligand_sdf)     as f: lig = f.read()
    v = py3Dmol.view(width=w, height=h)
    v.addModel(rec, "pdbqt"); v.setStyle({"model": 0}, {"cartoon": {"color": "spectrum"}})
    v.addModel(lig, "sdf");   v.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
    v.zoomTo({"model": 1})
    return v

for i, pose in enumerate(pose_paths):
    if pose is None:
        print(f"[{i}] no pose file (docking likely failed)"); continue
    print(f"\n[{i}]  aff={affinities[i]:+.2f}  smi={smiles_out[i]}")
    view_pose(receptor, pose).show()

In [ ]:
# Optional summary save (JSON, no binary objects). Trajectories regenerate from
# the same checkpoint + seed, so we don't persist them.
summary = {
    "target_name":       hps.target_name,
    "saved_model_path":  hps.saved_model_path,
    "smiles":            list(smiles_out),
    "affinities":        [float(a) for a in affinities],
    "rewards":           [float(r) for r in rewards],
    "pose_paths":        pose_paths,
    "receptor":          receptor,
}
out_path = os.path.join(PERSIST_DIR, f"{hps.target_name}_visualize_results.json")
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)
print("wrote", out_path)